## EA Pipeline Skeleton

In [49]:
from pathlib import Path
import pandas as pd
import numpy as np
import datetime
import random
import sys
import os
import glob
import csv

### PlaneModel defines a set of useful classes

In [38]:
from  PlaneModel import job,aircraft,staff,problem

### Import Data Generator and its aircraft instaces method

In [39]:
from generate_aircrafts_seed import generate_aircraft_instances

### Import EA Computing Functions and Depencies

In [40]:
from EA_Model import randSol, evaluate, timeMutate, mutate, copyG, contains, xo, tour, rip

### Generate CSV file with specified input parameters

In [41]:
generate_aircraft_instances(
    work_packages_file="work_packages.csv",  # Relative Path to WP input CSV
    output_dir="generated_aircrafts",        # Output directory name
    num_instances=10,                        # Number of CSV files to generate
    seed=20                                  # Fix the seed to 20 to ensure reproducibility
)

### Conditional Checking of Data Directory

In [42]:
def is_directory_empty(path_directory):
    """
    Check if a directory is empty.
    """
    path = Path(path_directory)
    # Check if the path exists and is a directory
    if not path.exists() or not path.is_dir():
        raise ValueError("Invalid directory path")
    # check if the directory is empty
    return not any(path.iterdir())

In [43]:
path_directory = "generated_aircrafts"
try:
    if is_directory_empty(path_directory):
        print("Directory is empty")
    else:
        print("sample aircrfat instances are generated")
except ValueError as e:
    print(e)
    sys.exit(1)
    

sample aircrfat instances are generated


### Reading Work Packages and Technicians files to DF

In [44]:
work_packages = pd.read_csv('work_packages.csv', index_col=0)
people = pd.read_csv('technicians.csv')

## Load Problem instance from the directory of database


In [45]:
# Path to the directory containing the CSV files
db_path = './generated_aircrafts'

# Get a list of all CSV files in the directory
csv_files = glob.glob(os.path.join(db_path, '*.csv'))

In [46]:
#create a dict to store the aircraft samples
aircraft_samples = {}

# Loop through each CSV file and read it into a DataFrame
for csv_file in csv_files:
    # Extract the filename without the directory and extension
    filename = os.path.splitext(os.path.basename(csv_file))[0]
    
    # Read the CSV file into a DataFrame
    df = pd.read_csv(csv_file)
    
    # Store the DataFrame in the dictionary with the filename as the key
    aircraft_samples[filename] = df

### Input Logs Code and CSV File

In [47]:
# Create logs directory if it doesn't exist
os.makedirs('logs', exist_ok=True)

wp_origin = pd.read_csv('work_packages.csv')

# map the WP numbers to minutes into the dictionary
wp_time_dict = wp_origin.set_index("WP number")["Minutes"].astype(int).to_dict()
wp_personnel_dict = wp_origin.set_index("WP number")["Number of Personnel"].astype(int).to_dict()


# List to store results for each file
results = []
    
# Loop through each DataFrame in the dictionary
for filename, df in aircraft_samples.items():
    
     # calculate total number of rows in the dataframe
    aircraft_num = len(df)
    
    # Total Turnaround Time (Column F)
    # Convert HH:MM to total minutes for summing
    tat = df['Turn Around Time'].str.split(':', expand=True).astype(int)
    total_minutes = (tat[0] * 60 + tat[1]).sum()
    total_tat = f"{total_minutes}"
    # total_hours, total_mins = divmod(total_minutes, 60)
    # total_tat = f"{total_hours}:{total_mins:02d}"
    
    # Average Turnaround Time
    avg_minutes = total_minutes / aircraft_num
    avg_hours, avg_mins = divmod(int(avg_minutes), 60)
    avg_tat = f"{avg_hours}:{avg_mins:02d}"
    
    # Total WPs (Column G)
    wp_counts = df['Work that needs to be carried out'].apply(lambda x: len(str(x).split(', ')))
    total_wps = wp_counts.sum()
    avg_wps = round(wp_counts.mean(), 2)  # New metric: average WPs
    
    total_work_minutes = 0
    total_personnel = 0
    for wp_list in df['Work that needs to be carried out']:
        wps = str(wp_list).split(', ')
        for wp in wps:
            wp = wp.strip()
            total_work_minutes += wp_time_dict.get(wp, 0)  # Default to 0 if WP not found
            total_personnel += wp_personnel_dict.get(wp, 0)
            
            
    # Average personnel per WP
    if total_wps > 0:
        avg_personnel_per_wp = round(total_personnel / total_wps, 2) 
    else:
        avg_personnel_per_wp = 0

    
    results.append({
        'Filename': filename,
        'Total Aircraft Serial Numbers': aircraft_num,
        'Total Turnaround Time Minutes': total_tat,
        'Average Turnaround Time': avg_tat,
        'Total Number WPs': total_wps,
        'Average Number WPs': avg_wps,
        'Total Work Minutes': total_work_minutes,
        'Average Personnel per WP': avg_personnel_per_wp,
    })
    
# Create a DataFrame from the results list
results_df = pd.DataFrame(results)


output_path = os.path.join('logs', 'output_result.csv')
results_df.to_csv(output_path, index=False)
print(f"Aircraft sample work summary saved to {output_path}")
 
    

Aircraft sample work summary saved to logs\output_result.csv


### Call the EA Model and Populate it with DataFrames of Aircraft Samples

#### Fine-tuning the EA outputs

In [ ]:
#create a logs directory if it doesn't exist
os.makedirs('logs', exist_ok=True)

#create a CSV output file with the summary of the aircraft samples and svae it in the logs directory
with open(os.path.join('logs', 'results.csv'), 'w', newline='') as csvfile:
    fieldnames = [ 'Problem ID', 
        'Run Number', 
        'Initial Best', 
        'Evals', 
        'Improved Fitness', 
        'Missing Staff']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    


for k , planes in aircraft_samples.items():
    instance = problem(people,planes,work_packages)
    
    print(f"Running for {k} instance")

    fitness_All = []
    for run in range(10):
        pop_size = 1500 
        budget = 1000

        best = None

        pop = []
        for c in range(pop_size):
            
            i = randSol(instance)
            f= evaluate(i, instance)[0]
            p = (f, i)
            pop.append(p)
            if len(pop) == 1:
                best = p
            if f < best[0]:
                best = p

        print(f"Run {run + 1} - Init")
        print(f"Initial best: {best[0]}")

        evals = 0
        
        while evals < budget:
            evals += 1
            if random.choice([True, False]):
                parent = tour(pop)
                ng = copyG(parent[1])
            else:
                ng = xo(tour(pop)[1], tour(pop)[1])
            
            mutate(ng, instance)
            nf = evaluate(ng, instance)[0]
            child = (nf, ng)

            toGo = rip(pop)
            if toGo[0] > child[0]:
                pop.remove(toGo)
                pop.append(child)
                if child[0] < best[0]:
                    best = (child[0], copyG(child[1]))
                    instance.reset()
                    r = evaluate(best[1], instance)
                    print(f"Evals: {evals}")
                    print(f"Improved ({r[0]}) Missing staff = {r[1]} Late = {r[2]} Late mins = {r[3]}")

        fitness_All.append(best[0])
        print(f"Run {run + 1} - Done: {best[0]}\n")
        
        #create a directory for the solutions if it doesn't exist
        os.makedirs("solutions", exist_ok=True)
        
        #Crrate a filename with path
        filename = os.path.join("solutions", f"{k}_run_{run + 1}_solution.txt")
        
        #write the solution to a txt file
        instance.reset()
        r = evaluate(best[1], instance)
        with open(filename, "w") as f:
            f.write(f"Fitness = {r[0]}\n")
            f.write(f"Missing staff: {r[1]}\n")
            f.write(f"Late Departures: {r[2]}\n")
            f.write(f"Late mins: {r[3]}\n")
            f.write(f"\n Plan : \n\n{instance}")
        
    # compute the average of fitness values in the population for 10 iterations
    np_avg_fitness = np.mean(fitness_All)
    avg_fitness = sum(fitness_All) / len(fitness_All)
    print(f"Average fitness over 10 runs: Math Method {avg_fitness} and Numpy Method {np_avg_fitness}")

    # compute standard deviation of fitness values in the population for 10 iterations
    std_dev = np.std(fitness_All)
    print(f"Standard Deviation: {std_dev}")

Running for aircrafts_sample_1 instance
Run 1 - Init
Initial best: 24
Run 1 - Done: 24

Run 2 - Init
Initial best: 25
Evals: 515
Improved (24) Missing staff = 4 Late = 2 Late mins = 20
Run 2 - Done: 24

Run 3 - Init
Initial best: 25
Evals: 847
Improved (15) Missing staff = 5 Late = 1 Late mins = 10
Run 3 - Done: 15

Run 4 - Init
Initial best: 17
Evals: 602
Improved (16) Missing staff = 6 Late = 1 Late mins = 10
Run 4 - Done: 16

Run 5 - Init
Initial best: 14
Run 5 - Done: 14

Run 6 - Init
Initial best: 33
Evals: 142
Improved (26) Missing staff = 6 Late = 2 Late mins = 20
Evals: 789
Improved (16) Missing staff = 6 Late = 1 Late mins = 10
Run 6 - Done: 16

Run 7 - Init
Initial best: 25
Evals: 191
Improved (24) Missing staff = 4 Late = 2 Late mins = 20
Evals: 543
Improved (17) Missing staff = 7 Late = 1 Late mins = 10
Run 7 - Done: 17

Run 8 - Init
Initial best: 7
Run 8 - Done: 7

Run 9 - Init
Initial best: 15
Run 9 - Done: 15

Run 10 - Init
Initial best: 27
Evals: 142
Improved (26) Missi